# Stage 5: Review Sentiment Export Layer

**Stakeholder:** Property Investment Analytics <br>
**Stage:** 5 of 5: Review Sentiment Export <br>
**Medallion Layer:** Export 📊 <br>
**Cities:** London · Manchester · Edinburgh · Bristol <br>
**Authors:** Adam <br>
**Last Updated:** June 2026

---

### Purpose

This notebook runs lexicon-based sentiment analysis (VADER) over every Airbnb review, joins each review to its listing's MSOA and LAD via the gold layer, then aggregates to neighbourhood level. Output includes average sentiment, review volume, a positive/negative/neutral breakdown, and a sample of representative reviews per area.

**Reads from:** `airbnb_app.clean.airbnb_reviews_{city}`, `airbnb_app.gold.airbnb_listings_{city}`  
**Writes to:** `airbnb_app.export.msoa_review_sentiment`, `airbnb_app.export.lad_review_sentiment`

> **Note:** VADER is a lexicon-based sentiment model — fast and runs entirely on the driver, no GPU required. It is tuned for short, informal text, which fits Airbnb reviews well, but is less accurate than a transformer model on nuanced or sarcastic language.  
> **Edinburgh:** included at LAD level only — no MSOA codes (Scotland uses Data Zones).  
> ⚠️ **Photon UNION ALL bug:** spot checks loop per-city — see notebook 03 for full explanation.

---

### How to Use

Run all cells in order. Notebooks 01–03 must have completed successfully (this notebook needs `airbnb_app.gold` for MSOA/LAD codes and `airbnb_app.clean.airbnb_reviews_{city}` for review text).

**Adjusting sample review count**  
Change `SAMPLE_REVIEWS_PER_AREA` in Section 0.

**Swapping sentiment models**  
The scoring logic is isolated in the `score_sentiment` function in Section 2 — replace its internals to swap in a transformer model later without changing the aggregation logic downstream.

---

### Flowchart

`airbnb_app.clean.airbnb_reviews_{city}` → `Join MSOA/LAD from Gold` → `VADER Sentiment Scoring` → `Aggregate to MSOA` → `Aggregate to LAD` → `Sample Representative Reviews` → `Write to airbnb_app.export`

## 0. Config

In [0]:
CLEAN_DB  = "airbnb_app.clean"
GOLD_DB   = "airbnb_app.gold"
EXPORT_DB = "airbnb_app.export"

CITIES = ["london", "manchester", "edinburgh", "bristol"]

# Sentiment thresholds (VADER compound score, -1 to +1)
POSITIVE_THRESHOLD = 0.05
NEGATIVE_THRESHOLD = -0.05

# Number of sample reviews to retain per area for qualitative context
SAMPLE_REVIEWS_PER_AREA = 3

# Minimum review length to include (filters out blank/non-English short reviews)
MIN_REVIEW_LENGTH = 10

# Cap reviews scored per listing — keeps area-level averages reliable
# while avoiding scoring every single review (major speed win)
MAX_REVIEWS_PER_LISTING = 5

## 1. Setup

In [0]:
%pip install vaderSentiment --quiet
%pip install vaderSentiment langdetect --quiet

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {EXPORT_DB}")
print(f"Schema ready: {EXPORT_DB}")

analyzer = SentimentIntensityAnalyzer()
export_log = []

## 2. Helpers

In [0]:
from langdetect import detect, LangDetectException

def is_english(text: str) -> bool:
    """Return True if text is detected as English, False otherwise (or on detection failure)."""
    if not isinstance(text, str) or len(text.strip()) < MIN_REVIEW_LENGTH:
        return False
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False


def score_sentiment(text: str) -> float:
    """
    Return VADER compound sentiment score for a single review.
    Compound score ranges from -1 (most negative) to +1 (most positive).
    Language filtering is handled separately by is_english() before this runs.
    """
    if not isinstance(text, str) or len(text.strip()) < MIN_REVIEW_LENGTH:
        return None
    return analyzer.polarity_scores(text)["compound"]


def classify_sentiment(score: float) -> str:
    """Bucket a compound sentiment score into positive / neutral / negative."""
    if score is None:
        return None
    if score >= POSITIVE_THRESHOLD:
        return "positive"
    if score <= NEGATIVE_THRESHOLD:
        return "negative"
    return "neutral"


def sample_reviews(group: pd.DataFrame, n: int) -> str:
    """
    Pick representative sample reviews for an area: one of the most positive,
    one of the most negative, and one closest to the area's average sentiment.
    Returned as a single ' | '-delimited string for easy export.
    """
    if group.empty:
        return None
    sorted_group = group.sort_values("sentiment_score")
    picks = []
    if len(sorted_group) >= 1:
        picks.append(sorted_group.iloc[0])
    if len(sorted_group) >= 2:
        picks.append(sorted_group.iloc[-1])
    if len(sorted_group) >= 3 and n >= 3:
        mean_score = group["sentiment_score"].mean()
        closest_idx = (group["sentiment_score"] - mean_score).abs().idxmin()
        picks.append(group.loc[closest_idx])
    snippets = [p["comments"][:200] for p in picks[:n]]
    return " | ".join(snippets)


print("Helpers loaded.")

---
## Part A — Score Reviews and Attach Geography

In [0]:
scored_frames = []

for city in CITIES:
    print(f"\n{'='*50}")
    print(f"{city.upper()}")
    print(f"{'='*50}")

    try:
        # Load reviews (clean layer) and listing geography (gold layer)
        reviews_pd = spark.table(f"{CLEAN_DB}.airbnb_reviews_{city}").toPandas()
        geo_pd = (
            spark.table(f"{GOLD_DB}.airbnb_listings_{city}")
            .select("id", "lad_code", "lad_name", "msoa_code", "msoa_name")
            .toPandas()
        )

        print(f"  Loaded {len(reviews_pd):,} reviews")

        # Join reviews to listing geography on listing_id
        reviews_pd = reviews_pd.merge(
            geo_pd, left_on="listing_id", right_on="id", how="left"
        )
        matched = reviews_pd["lad_code"].notna().sum()
        print(f"  Matched to geography: {matched:,} / {len(reviews_pd):,}")

        # Cap reviews per listing before any scoring — major speed win,
        # area-level averages stay reliable with a random sample per listing
        reviews_pd = (
            reviews_pd.groupby("listing_id", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), MAX_REVIEWS_PER_LISTING), random_state=42))
        )
        print(f"  Capped to {len(reviews_pd):,} reviews (max {MAX_REVIEWS_PER_LISTING}/listing)")

        # Filter to English-language reviews only — VADER is English-only
        # and scoring non-English text produces meaningless results
        reviews_pd["is_english"] = reviews_pd["comments"].apply(is_english)
        non_english_count = (~reviews_pd["is_english"]).sum()
        print(f"  Filtered out {non_english_count:,} non-English reviews")
        reviews_pd = reviews_pd[reviews_pd["is_english"]].drop(columns="is_english")

        # Score sentiment
        reviews_pd["sentiment_score"] = reviews_pd["comments"].apply(score_sentiment)
        reviews_pd["sentiment_label"] = reviews_pd["sentiment_score"].apply(classify_sentiment)
        reviews_pd = reviews_pd.dropna(subset=["sentiment_score"])

        reviews_pd["city"] = city
        scored_frames.append(reviews_pd)

        print(f"  Scored: {len(reviews_pd):,} reviews")
        export_log.append({"city": city, "status": "ok", "reviews_scored": len(reviews_pd)})

    except Exception as e:
        print(f"  ✗ {city} — {e}")
        export_log.append({"city": city, "status": "error", "error": str(e)})

all_reviews_pd = pd.concat(scored_frames, ignore_index=True)
print(f"\nTotal scored reviews across all cities: {len(all_reviews_pd):,}")

---
## Part B — Aggregate to MSOA Level (England & Wales)

In [0]:
msoa_reviews = all_reviews_pd[all_reviews_pd["msoa_code"].notna()].copy()

msoa_agg = (
    msoa_reviews.groupby(["msoa_code", "msoa_name", "lad_code", "lad_name", "city"])
    .agg(
        review_count=("sentiment_score", "count"),
        avg_sentiment_score=("sentiment_score", "mean"),
    )
    .reset_index()
)

# Positive/negative/neutral percentage breakdown
label_pct = (
    msoa_reviews.groupby(["msoa_code", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)
label_pct = label_pct.div(label_pct.sum(axis=1), axis=0).round(4) * 100
label_pct = label_pct.rename(columns={
    "positive": "pct_positive", "neutral": "pct_neutral", "negative": "pct_negative"
}).reset_index()

msoa_agg = msoa_agg.merge(label_pct, on="msoa_code", how="left")

# Sample representative reviews per MSOA
samples = (
    msoa_reviews.groupby("msoa_code")
    .apply(lambda g: sample_reviews(g, SAMPLE_REVIEWS_PER_AREA))
    .reset_index(name="sample_reviews")
)
msoa_agg = msoa_agg.merge(samples, on="msoa_code", how="left")

msoa_agg["avg_sentiment_score"] = msoa_agg["avg_sentiment_score"].round(4)

print(f"MSOA review sentiment table: {msoa_agg.shape}")
msoa_agg.head(5)

---
## Part C — Aggregate to LAD Level (All Cities)

In [0]:
lad_reviews = all_reviews_pd[all_reviews_pd["lad_code"].notna()].copy()

lad_agg = (
    lad_reviews.groupby(["lad_code", "lad_name", "city"])
    .agg(
        review_count=("sentiment_score", "count"),
        avg_sentiment_score=("sentiment_score", "mean"),
    )
    .reset_index()
)

label_pct_lad = (
    lad_reviews.groupby(["lad_code", "sentiment_label"])
    .size()
    .unstack(fill_value=0)
)
label_pct_lad = label_pct_lad.div(label_pct_lad.sum(axis=1), axis=0).round(4) * 100
label_pct_lad = label_pct_lad.rename(columns={
    "positive": "pct_positive", "neutral": "pct_neutral", "negative": "pct_negative"
}).reset_index()

lad_agg = lad_agg.merge(label_pct_lad, on="lad_code", how="left")

samples_lad = (
    lad_reviews.groupby("lad_code")
    .apply(lambda g: sample_reviews(g, SAMPLE_REVIEWS_PER_AREA))
    .reset_index(name="sample_reviews")
)
lad_agg = lad_agg.merge(samples_lad, on="lad_code", how="left")

lad_agg["avg_sentiment_score"] = lad_agg["avg_sentiment_score"].round(4)

print(f"LAD review sentiment table: {lad_agg.shape}")
lad_agg.head(5)

---
## Part D — Write Export Tables

In [0]:
def write_export(pdf: pd.DataFrame, table_name: str, description: str):
    for c in pdf.select_dtypes(include="float32").columns:
        pdf[c] = pdf[c].astype("float64")

    sdf = spark.createDataFrame(pdf)
    sdf = sdf.withColumn("_export_created_at", F.current_timestamp())

    tgt = f"{EXPORT_DB}.{table_name}"
    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tgt)
    )
    rows = sdf.count()
    print(f"  ✓ {tgt} — {rows:,} rows  ({description})")


print("Writing export tables...")
write_export(msoa_agg, "msoa_review_sentiment", "MSOA-level review sentiment aggregation")
write_export(lad_agg,  "lad_review_sentiment",  "LAD-level review sentiment aggregation (all cities)")
print("\nExport writes complete.")

---
## Part E — Spot Checks

In [0]:
# Top 10 MSOAs by average sentiment, minimum 10 reviews
spark.sql("""
    SELECT city, msoa_name, lad_name, review_count,
           avg_sentiment_score, pct_positive, pct_negative
    FROM airbnb_app.export.msoa_review_sentiment
    WHERE review_count >= 10
    ORDER BY avg_sentiment_score DESC
    LIMIT 10
""").display()

In [0]:
# Bottom 10 MSOAs by average sentiment, minimum 10 reviews
spark.sql("""
    SELECT city, msoa_name, lad_name, review_count,
           avg_sentiment_score, pct_positive, pct_negative
    FROM airbnb_app.export.msoa_review_sentiment
    WHERE review_count >= 10
    ORDER BY avg_sentiment_score ASC
    LIMIT 10
""").display()

In [0]:
# LAD-level summary across all four cities, including Edinburgh
spark.sql("""
    SELECT city, lad_name, review_count, avg_sentiment_score,
           pct_positive, pct_neutral, pct_negative
    FROM airbnb_app.export.lad_review_sentiment
    ORDER BY city, avg_sentiment_score DESC
""").display()

In [0]:
# Ingestion / scoring summary log
summary = pd.DataFrame(export_log)
display(summary)

print("\nReview sentiment export complete.")

In [0]:
import base64

def make_download_link(csv_path: str, filename: str) -> str:
    with open(csv_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'<a href="data:text/csv;base64,{b64}" download="{filename}">{filename}</a>'

VOLUME_PATH = "/Volumes/airbnb_app/gold/exports/"

msoa_agg.to_csv(f"{VOLUME_PATH}msoa_review_sentiment.csv", index=False)
lad_agg.to_csv(f"{VOLUME_PATH}lad_review_sentiment.csv", index=False)

displayHTML(
    make_download_link(f"{VOLUME_PATH}msoa_review_sentiment.csv", "msoa_review_sentiment.csv")
    + "<br><br>" +
    make_download_link(f"{VOLUME_PATH}lad_review_sentiment.csv", "lad_review_sentiment.csv")
)

## Notes

- **VADER** is lexicon-based and rule-driven — fast, no GPU, and well suited to short informal text like Airbnb reviews. It will under-perform on sarcasm, idioms, and non-English reviews.
- **Sentiment thresholds:** compound score ≥ 0.05 = positive, ≤ -0.05 = negative, otherwise neutral. These are VADER's standard defaults — adjust in Section 0 if needed.
- **Sample reviews** are selected to show range: the most negative, the most positive, and one closest to the area average — giving qualitative colour alongside the aggregate score.
- **Geography join** uses `listing_id` to pull `lad_code`/`msoa_code` from the gold layer — reviews inherit their listing's location.
- **Edinburgh** appears in the LAD export only, consistent with the rest of the pipeline (no MSOA codes available for Scotland).
- **Minimum review length** of 10 characters filters out blank or placeholder reviews (e.g. "N/A", ".") that would otherwise skew sentiment scores.
- **Next step:** join `msoa_review_sentiment` to `msoa_investment_summary` on `msoa_code` in the Streamlit frontend to combine investment yield with guest satisfaction signal.